# Text Analytics 2026 — Assignment 1

## N-gram Language Models με Laplace και Kneser–Ney Smoothing

**Φοιτητές:** Νέγκι Χότζα ΑΜ: p3352524 — Γιώργος Κωστόπουλος ΑΜ: p3352520
**Στόχος:** Υλοποίηση και σύγκριση bigram και trigram language models σε corpus προτάσεων.

Στο notebook υλοποιούνται τέσσερα μοντέλα:

| Smoothing | Bigram | Trigram |
|---|---:|---:|
| Laplace smoothing | ✓ | ✓ |
| Kneser–Ney smoothing | ✓ | ✓ |

Η βασική σχεδίαση του κώδικα είναι:

```text
NGramLM
├── LaplaceLM
└── KneserNeyLM
```

Η `NGramLM` περιέχει την κοινή n-gram λογική, ενώ οι δύο υποκλάσεις αλλάζουν μόνο τον τρόπο υπολογισμού των πιθανοτήτων.

Για το autocomplete υλοποιούμε δύο deterministic decoding methods:

- **Greedy decoding**
- **Beam search** με length normalization

Η επιλογή deterministic μεθόδων εξασφαλίζει ότι τα αποτελέσματα είναι **πλήρως αναπαραγώγιμα** και άμεσα συγκρίσιμα μεταξύ μοντέλων.


## 1. Θεωρητική περιγραφή του προβλήματος

Ένα language model προσπαθεί να εκτιμήσει την πιθανότητα μιας ακολουθίας λέξεων:

$$
P(w_1, w_2, \ldots, w_N)
$$

Επειδή η πλήρης πιθανότητα εξαρτάται από όλο το προηγούμενο ιστορικό, χρησιμοποιούμε την n-gram προσέγγιση.

### Bigram model

Στο bigram μοντέλο θεωρούμε ότι κάθε λέξη εξαρτάται μόνο από την αμέσως προηγούμενη λέξη:

$$
P(w_1, \ldots, w_N) \approx \prod_{i=1}^{N} P(w_i \mid w_{i-1})
$$

### Trigram model

Στο trigram μοντέλο θεωρούμε ότι κάθε λέξη εξαρτάται από τις δύο προηγούμενες λέξεις:

$$
P(w_1, \ldots, w_N) \approx \prod_{i=1}^{N} P(w_i \mid w_{i-2}, w_{i-1})
$$

Στην πράξη δεν υπολογίζουμε απευθείας το γινόμενο, γιατί οι πιθανότητες είναι μικροί αριθμοί και το γινόμενο πολλών μικρών αριθμών μπορεί να οδηγήσει σε αριθμητικό underflow. Για αυτό χρησιμοποιούμε άθροισμα λογαρίθμων:

$$
\log_2 P(w_1, \ldots, w_N) = \sum_{i=1}^{N} \log_2 P(w_i \mid context_i)
$$


## 2. Εγκατάσταση και imports

Χρησιμοποιούμε το NLTK μόνο για:

- φόρτωση του Brown corpus,
- έτοιμο sentence splitting/tokenization του corpus,
- βοηθητική παραγωγή n-grams.

Η εκτίμηση πιθανοτήτων, το smoothing, το cross-entropy, το perplexity και το autocomplete υλοποιούνται στον δικό μας κώδικα.


In [ ]:
# Εγκατάσταση/ενημέρωση του NLTK.
# Αν το περιβάλλον έχει ήδη εγκατεστημένο το nltk, αυτό το cell δεν χρειάζεται να αλλάξει κάτι.
!pip install -U nltk -q

In [ ]:
# Βασικές βιβλιοθήκες.
import random
import math
from collections import Counter, defaultdict

# NLTK: το χρησιμοποιούμε για το Brown corpus και για τη βοηθητική συνάρτηση ngrams.
import nltk
from nltk.corpus import brown
from nltk.util import ngrams


## 3. Φόρτωση του Brown corpus από το NLTK

Κατεβάζουμε το Brown corpus και επιλέγουμε την κατηγορία `lore`, την οποία θα χρησιμοποιήσουμε για όλα τα πειράματα.


In [ ]:
# Κατεβάζουμε το Brown corpus, αν δεν υπάρχει ήδη τοπικά.
nltk.download("brown", quiet=True)

# Εκτύπωση των κατηγοριών του Brown corpus και του πλήθους προτάσεων σε καθεμία.
print("Brown categories and number of sentences:\n")

for cat in brown.categories():
    sentence_count = len(brown.sents(categories=cat))
    print("-- {}: {}".format(cat, sentence_count))

In [ ]:
# Φορτώνουμε τις προτάσεις της κατηγορίας 'lore'.
# Κάθε πρόταση είναι λίστα από tokens.
sentences_tokenized = [
    list(sent)
    for sent in brown.sents(categories="lore")
]

print("Number of sentences:", len(sentences_tokenized))
print()

# Εμφανίζουμε μερικά παραδείγματα προτάσεων.
for sent in sentences_tokenized[:5]:
    print(sent)
    print("________________")


## 4. Προεπεξεργασία δεδομένων

Τα βασικά preprocessing βήματα είναι:

1. μετατροπή όλων των tokens σε πεζά,
2. τυχαίο shuffle των προτάσεων,
3. διαχωρισμός σε training/development/test (80% / 10% / 10%),
4. δημιουργία vocabulary μόνο από το training set,
5. αντικατάσταση όλων των OOV λέξεων με `*UNK*`.

Το vocabulary φτιάχνεται **μόνο από το training subset**, ώστε να μην υπάρχει πληροφορία από development ή test κατά την εκπαίδευση.


In [ ]:
# Μετατροπή όλων των tokens σε πεζά.
# Έτσι, π.χ. "The" και "the" αντιμετωπίζονται ως η ίδια λέξη.
sentences_lower = [
    [token.lower() for token in sent]
    for sent in sentences_tokenized
]

print("Example after lowercasing:")
print(sentences_lower[0])


In [ ]:
# Θέτουμε seed για να είναι αναπαραγώγιμο το shuffle.
random.seed(0)

# Κάνουμε shuffle στις προτάσεις, όχι στις λέξεις μέσα στις προτάσεις.
random.shuffle(sentences_lower)

# Ορίζουμε τα ποσοστά train/dev/test.
train_ratio = 0.8
dev_ratio = 0.1
test_ratio = 0.1

N_sentences = len(sentences_lower)

train_end = int(train_ratio * N_sentences)
dev_end = int((train_ratio + dev_ratio) * N_sentences)

train_sentences_raw = sentences_lower[:train_end]
dev_sentences_raw = sentences_lower[train_end:dev_end]
test_sentences_raw = sentences_lower[dev_end:]

print("Train size:", len(train_sentences_raw))
print("Dev size:  ", len(dev_sentences_raw))
print("Test size: ", len(test_sentences_raw))


## 5. Vocabulary και OOV λέξεις

Η εκφώνηση ζητά να κρατήσουμε στο vocabulary μόνο λέξεις που εμφανίζονται τουλάχιστον κάποιες φορές στο training set. Εδώ χρησιμοποιούμε:

$$
\text{MIN\_COUNT} = 10
$$

Άρα:

$$
V = \{w : count_{train}(w) \geq 10\}
$$

Όλες οι υπόλοιπες λέξεις αντικαθίστανται με το ειδικό token:

$$
*UNK*
$$

Επίσης προσθέτουμε στο vocabulary το `*end*`, γιατί το μοντέλο πρέπει να μπορεί να προβλέπει το τέλος πρότασης.


In [ ]:
# Ειδικά tokens που ζητάει η εκφώνηση.
UNK = "*UNK*"
START = "*start*"
START1 = "*start1*"
START2 = "*start2*"
END = "*end*"

# Ελάχιστος αριθμός εμφανίσεων στο training set για να μπει μια λέξη στο vocabulary.
MIN_COUNT = 10

# Μετράμε μόνο λέξεις από το training set.
word_counts = Counter()
for sent in train_sentences_raw:
    word_counts.update(sent)

# Κρατάμε μόνο λέξεις που εμφανίζονται τουλάχιστον MIN_COUNT φορές.
vocab = {
    word for word, count in word_counts.items()
    if count >= MIN_COUNT
}

# Προσθέτουμε τα ειδικά tokens που πρέπει να μπορούν να προβλεφθούν.
vocab.add(UNK)
vocab.add(END)

print("Vocabulary size:", len(vocab))
print("First 50 vocabulary items:")
print(list(vocab)[:50])

### Συνάρτηση `replace_oov`

Η `replace_oov` δέχεται μια λίστα από προτάσεις και ένα vocabulary $V$, και αντικαθιστά κάθε λέξη που δεν ανήκει στο $V$ με το ειδικό token `*UNK*`:

$$
\text{replace}(w) =
\begin{cases}
w, & \text{αν } w \in V \\
*UNK*, & \text{αλλιώς}
\end{cases}
$$

Η ίδια συνάρτηση εφαρμόζεται σε train, dev και test, ώστε όλα τα subsets να μοιράζονται ακριβώς το ίδιο vocabulary και να μην υπάρχουν παράξενα OOV tokens μόνο στο test set.


In [ ]:
def replace_oov(sentences, vocab):
    """
    Αντικαθιστά κάθε λέξη που δεν υπάρχει στο vocabulary με το *UNK*.
    """
    new_sentences = []
    for sent in sentences:
        new_sent = [
            token if token in vocab else UNK
            for token in sent
        ]
        new_sentences.append(new_sent)
    return new_sentences


train_sentences = replace_oov(train_sentences_raw, vocab)
dev_sentences = replace_oov(dev_sentences_raw, vocab)
test_sentences = replace_oov(test_sentences_raw, vocab)

print("Example sentence after OOV replacement:")
print(train_sentences[0])

## 6. Start και end pseudo-tokens

Η εκφώνηση ζητά:

- για bigram model: κάθε πρόταση να ξεκινά με `*start*`,
- για trigram model: κάθε πρόταση να ξεκινά με `*start1*`, `*start2*`,
- κάθε πρόταση να τελειώνει με `*end*`.

Άρα, για παράδειγμα, η πρόταση:

```text
['the', 'man', 'left']
```

γίνεται στο bigram:

```text
['*start*', 'the', 'man', 'left', '*end*']
```

και στο trigram:

```text
['*start1*', '*start2*', 'the', 'man', 'left', '*end*']
```

Δεν προβλέπουμε τα start tokens, γιατί τα βάζουμε τεχνητά στην αρχή κάθε πρότασης. Προβλέπουμε όμως το `*end*`, γιατί το μοντέλο πρέπει να μάθει πότε τελειώνει μια πρόταση.


## 7. Γενική κλάση `NGramLM`

Η `NGramLM` είναι η κοινή βάση για όλα τα n-gram μοντέλα.

Περιέχει:

- την τιμή του `n`,
- το κοινό vocabulary,
- τα n-gram counts,
- τα context counts,
- τη λογική για προσθήκη start/end pseudo-tokens,
- τη συνάρτηση `log_prob`.

Δεν υλοποιεί συγκεκριμένο smoothing. Αυτό γίνεται στις υποκλάσεις `LaplaceLM` και `KneserNeyLM`.

Η μέγιστη πιθανοφάνεια (Maximum Likelihood Estimate) για n-gram με context $h$ είναι:

$$
P_{MLE}(w \mid h) = \frac{count(h, w)}{count(h)}
$$

αλλά δεν την χρησιμοποιούμε άμεσα, γιατί δίνει πιθανότητα 0 σε άγνωστα n-grams. Για αυτό κάθε υποκλάση εφαρμόζει διαφορετικό smoothing.


## 8. Laplace smoothing

Η μέγιστη πιθανοφάνεια θα έδινε:

$$
P_{MLE}(w \mid h) = \frac{count(h, w)}{count(h)}
$$

Το πρόβλημα είναι ότι αν ένα n-gram δεν εμφανιστεί στο training set, τότε έχει πιθανότητα 0. Αυτό είναι κακό για το perplexity, γιατί ένα μόνο μηδενικό μηδενίζει την πιθανότητα ολόκληρου του test corpus.

Με **add-one (Laplace) smoothing** προσθέτουμε 1 σε κάθε πιθανό n-gram:

$$
P_{Laplace}(w \mid h) = \frac{count(h, w) + 1}{count(h) + |V|}
$$

Πιο γενικά, με add-$\alpha$ smoothing:

$$
P(w \mid h) = \frac{count(h, w) + \alpha}{count(h) + \alpha \cdot |V|}
$$

Στην εργασία κρατάμε:

$$
\alpha = 1
$$

ώστε να είναι καθαρό Laplace/add-one smoothing.


## 9. Kneser–Ney smoothing

Το Kneser–Ney είναι πιο ισχυρό smoothing από το Laplace, επειδή δεν κοιτάει μόνο πόσες φορές εμφανίστηκε μια λέξη, αλλά και σε πόσα διαφορετικά contexts εμφανίστηκε.

### Bigram Kneser–Ney

Για bigram model:

$$
P_{KN}(w \mid h) =
\frac{\max(count(h,w) - D, 0)}{count(h)} +
\lambda(h) \cdot P_{cont}(w)
$$

όπου το **backoff weight** είναι:

$$
\lambda(h) = \frac{D \cdot N_{1+}(h, \bullet)}{count(h)}
$$

και η **continuation probability** είναι:

$$
P_{cont}(w) = \frac{N_{1+}(\bullet, w)}{N_{1+}(\bullet, \bullet)}
$$

Ερμηνεία των συμβόλων:

- $D$ είναι το **discount** (αφαιρείται από κάθε count),
- $N_{1+}(h, \bullet)$: πόσες **διαφορετικές** λέξεις έχουν εμφανιστεί μετά το context $h$,
- $N_{1+}(\bullet, w)$: σε πόσα **διαφορετικά** προηγούμενα contexts έχει εμφανιστεί η λέξη $w$,
- $N_{1+}(\bullet, \bullet)$: ο συνολικός αριθμός διαφορετικών bigrams στο training set.

### Trigram Kneser–Ney

Για trigram model κάνουμε **interpolated backoff** στο bigram KN μοντέλο:

$$
P_{KN}(w \mid u,v) =
\frac{\max(count(u,v,w) - D, 0)}{count(u,v)} +
\lambda(u,v) \cdot P_{KN}(w \mid v)
$$

όπου:

$$
\lambda(u,v) = \frac{D \cdot N_{1+}(u,v,\bullet)}{count(u,v)}
$$

Στο development set δοκιμάζουμε διάφορες τιμές του $D \in \{0.25, 0.5, 0.75, 0.9\}$ και κρατάμε αυτή με το μικρότερο perplexity.


### Υλοποίηση των κλάσεων `NGramLM`, `LaplaceLM`, `KneserNeyLM`

Στο επόμενο code cell υλοποιούμε τρεις κλάσεις:

- **`NGramLM`** — κρατά τα n-gram counts $count(w_1, \ldots, w_n)$ και τα context counts $count(w_1, \ldots, w_{n-1})$. Υλοποιεί επίσης την `log_prob`, που μετατρέπει πιθανότητα σε $\log_2$:

  $$
  \log_2 P(w \mid h)
  $$

- **`LaplaceLM`** (υποκλάση) — υλοποιεί τον τύπο:

  $$
  P_{Laplace}(w \mid h) = \frac{count(h, w) + \alpha}{count(h) + \alpha \cdot |V|}
  $$

- **`KneserNeyLM`** (υποκλάση) — υλοποιεί τον bigram και trigram KN τύπο που παρουσιάστηκε παραπάνω, μαζί με τους πίνακες $N_{1+}(h, \bullet)$ και $N_{1+}(\bullet, w)$.


In [ ]:
class NGramLM:
    """
    Βασική κλάση για n-gram language models.

    Περιέχει μόνο την κοινή λογική:
    - αποθήκευση του n και του vocabulary,
    - προσθήκη start/end pseudo-tokens,
    - μέτρηση n-grams και contexts,
    - αντικατάσταση άγνωστων λέξεων με *UNK*,
    - υπολογισμό log2 πιθανότητας.

    Η πραγματική πιθανότητα P(word | context) ορίζεται στις υποκλάσεις.
    """

    def __init__(self, n, vocab):
        if n not in {2, 3}:
            raise ValueError("This assignment uses only bigram and trigram models.")
        self.n = n
        self.vocab = vocab
        self.V = len(vocab)
        # n-gram counts, π.χ. για trigram: ('the', 'old', 'man').
        self.ngram_counts = Counter()
        # context counts μήκους n-1, π.χ. για trigram: ('the', 'old').
        self.context_counts = Counter()

    def add_sentence_tokens(self, sent):
        """Προσθέτει τα κατάλληλα start/end tokens ανάλογα με το n."""
        if self.n == 2:
            return [START] + sent + [END]
        return [START1, START2] + sent + [END]

    def train(self, sentences):
        """Εκπαιδεύει το μοντέλο μετρώντας n-grams και contexts."""
        for sent in sentences:
            tokens = self.add_sentence_tokens(sent)
            self.ngram_counts.update(ngrams(tokens, self.n))
            self.context_counts.update(ngrams(tokens, self.n - 1))

    def _fix_word(self, word):
        """Αν μια λέξη δεν είναι στο vocab, την αντικαθιστούμε με *UNK*."""
        if word in {START, START1, START2, END}:
            return word
        return word if word in self.vocab else UNK

    def _fix_context(self, context):
        """Εφαρμόζει την OOV λογική σε όλες τις λέξεις του context."""
        return tuple(self._fix_word(w) for w in context)

    def prob(self, word, context):
        """Υπολογίζει P(word | context). Ορίζεται στις υποκλάσεις."""
        raise NotImplementedError

    def log_prob(self, word, context):
        """Υπολογίζει log2 P(word | context) με προστασία από p=0."""
        p = self.prob(word, context)
        if p <= 0:
            p = 1e-12
        return math.log2(p)


class LaplaceLM(NGramLM):
    """
    Bigram/trigram μοντέλο με Laplace (add-alpha) smoothing.

    Τύπος:
        P(w | h) = (count(h, w) + alpha) / (count(h) + alpha * |V|)
    """

    def __init__(self, n, vocab, alpha=1.0):
        super().__init__(n, vocab)
        self.alpha = alpha

    def prob(self, word, context):
        word = self._fix_word(word)
        context = self._fix_context(context)
        ngram = context + (word,)
        numerator = self.ngram_counts[ngram] + self.alpha
        denominator = self.context_counts[context] + self.alpha * self.V
        return numerator / denominator


class KneserNeyLM(NGramLM):
    """
    Bigram/trigram μοντέλο με interpolated Kneser-Ney smoothing.

    Bigram:
        P_KN(w | h)   = max(c(h,w)-D, 0)/c(h)   + lambda(h)   * P_cont(w)
    Trigram:
        P_KN(w | u,v) = max(c(u,v,w)-D, 0)/c(u,v) + lambda(u,v) * P_KN(w | v)
    """

    def __init__(self, n, vocab, discount=0.75, lower_model=None):
        super().__init__(n, vocab)
        self.discount = discount
        # Trigram KN χρειάζεται ήδη εκπαιδευμένο bigram KN ως lower-order μοντέλο.
        self.lower_model = lower_model
        # N_1+(context, *): διαφορετικές λέξεις που ακολουθούν κάθε context.
        self.following_words = defaultdict(set)
        # Για bigram KN: N_1+(*, word) -> διαφορετικά previous words ανά word.
        self.previous_words = defaultdict(set)
        # N_1+(*, *): συνολικός αριθμός διαφορετικών bigrams.
        self.total_unique_bigrams = 0

    def train(self, sentences):
        super().train(sentences)
        for ng, count in self.ngram_counts.items():
            if count == 0:
                continue
            context = ng[:-1]
            word = ng[-1]
            self.following_words[context].add(word)
            if self.n == 2:
                previous_word = ng[0]
                self.previous_words[word].add(previous_word)
        if self.n == 2:
            self.total_unique_bigrams = len(self.ngram_counts)

    def _backoff_context(self, context):
        """
        Μετατρέπει trigram context σε bigram context για backoff.
        Ειδική περίπτωση αρχής πρότασης:
            ('*start1*', '*start2*') -> ('*start*',)
        """
        if context == (START1, START2):
            return (START,)
        return (context[-1],)

    def continuation_prob(self, word):
        """P_cont(word) = N_1+(*, word) / N_1+(*, *)."""
        if self.total_unique_bigrams == 0:
            return 0.0
        return len(self.previous_words[word]) / self.total_unique_bigrams

    def prob(self, word, context):
        word = self._fix_word(word)
        context = self._fix_context(context)
        count_context = self.context_counts[context]

        # --- Bigram KN ---
        if self.n == 2:
            if count_context == 0:
                return self.continuation_prob(word)
            ngram = context + (word,)
            count_ngram = self.ngram_counts[ngram]
            discounted = max(count_ngram - self.discount, 0) / count_context
            backoff_weight = self.discount * len(self.following_words[context]) / count_context
            return discounted + backoff_weight * self.continuation_prob(word)

        # --- Trigram KN ---
        if self.lower_model is None:
            raise ValueError("Trigram KN needs a trained bigram lower_model.")
        lower_context = self._backoff_context(context)
        if count_context == 0:
            return self.lower_model.prob(word, lower_context)
        ngram = context + (word,)
        count_ngram = self.ngram_counts[ngram]
        discounted = max(count_ngram - self.discount, 0) / count_context
        backoff_weight = self.discount * len(self.following_words[context]) / count_context
        return discounted + backoff_weight * self.lower_model.prob(word, lower_context)


## 10. Cross-entropy και Perplexity

Για το test corpus υπολογίζουμε το άθροισμα των log probabilities:

$$
\sum_{i=1}^{N} \log_2 P(w_i \mid context_i)
$$

Το **cross-entropy** είναι:

$$
H = -\frac{1}{N} \sum_{i=1}^{N} \log_2 P(w_i \mid context_i)
$$

και το **perplexity** είναι:

$$
PP = 2^{H}
$$

Το $N$ είναι ο αριθμός των tokens που προβλέπουμε. Σύμφωνα με την εκφώνηση:

- δεν προβλέπουμε ούτε μετράμε τα start pseudo-tokens,
- προβλέπουμε και μετράμε το `*end*`,
- άρα το `*end*` συμμετέχει κανονικά στο $N$.

### Συνάρτηση `evaluate_ngram`

Η `evaluate_ngram` τρέχει σε ένα corpus προτάσεων:

1. προσθέτει τα start/end pseudo-tokens μέσω της `add_sentence_tokens`,
2. για κάθε token από τη θέση $n-1$ και μετά, υπολογίζει $\log_2 P(w_i \mid context_i)$,
3. αθροίζει τα log probabilities και τα tokens,
4. επιστρέφει το $(H, PP)$.


In [ ]:
def evaluate_ngram(model, sentences):
    """
    Υπολογίζει cross-entropy και perplexity για ένα n-gram μοντέλο.

    Δεν προβλέπουμε start pseudo-tokens.
    Προβλέπουμε όμως το *end*, άρα το μετράμε και στο total_tokens.
    """
    total_log_prob = 0.0
    total_tokens = 0

    for sent in sentences:
        tokens = model.add_sentence_tokens(sent)
        # Ξεκινάμε από το index n-1, ώστε τα start tokens να είναι μόνο context.
        for i in range(model.n - 1, len(tokens)):
            context = tuple(tokens[i - model.n + 1:i])
            word = tokens[i]
            total_log_prob += model.log_prob(word, context)
            total_tokens += 1

    cross_entropy = -total_log_prob / total_tokens
    perplexity = 2 ** cross_entropy
    return cross_entropy, perplexity


## 11. Εκπαίδευση μοντέλων

Εκπαιδεύουμε τέσσερα μοντέλα:

1. Bigram με Laplace smoothing,
2. Trigram με Laplace smoothing,
3. Bigram με Kneser–Ney smoothing,
4. Trigram με Kneser–Ney smoothing.

Για το Laplace δεν κάνουμε tuning, επειδή χρησιμοποιούμε καθαρό add-one smoothing:

$$
\alpha = 1
$$

Για το Kneser–Ney κάνουμε tuning μόνο στο discount $D$ πάνω στο development set, ψάχνοντας πάνω στο σύνολο:

$$
D \in \{0.25, 0.5, 0.75, 0.9\}
$$

και κρατάμε τιμή με το μικρότερο dev perplexity.

### Συναρτήσεις `train_laplace` και `train_best_kneser_ney`

- **`train_laplace(n, train_sentences, vocab)`** — δημιουργεί `LaplaceLM` με $\alpha = 1$ και τρέχει `train`.
- **`train_best_kneser_ney(...)`** — δοκιμάζει διάφορα $D$, υπολογίζει dev perplexity και επιστρέφει το μοντέλο με το ελάχιστο.


In [ ]:
def train_laplace(n, train_sentences, vocab):
    """Εκπαιδεύει ένα Laplace n-gram μοντέλο με alpha=1."""
    model = LaplaceLM(n=n, vocab=vocab, alpha=1.0)
    model.train(train_sentences)
    return model


def train_best_kneser_ney(n, train_sentences, dev_sentences, vocab, lower_model=None):
    """
    Εκπαιδεύει Kneser-Ney μοντέλα με διαφορετικά D και κρατά αυτό
    με το χαμηλότερο dev perplexity.
    """
    discounts = [0.25, 0.5, 0.75, 0.9]
    best_model = None
    best_pp = float("inf")

    for D in discounts:
        model = KneserNeyLM(n=n, vocab=vocab, discount=D, lower_model=lower_model)
        model.train(train_sentences)
        _, pp = evaluate_ngram(model, dev_sentences)
        print(f"Kneser-Ney n={n}, D={D}: dev perplexity = {pp:.3f}")
        if pp < best_pp:
            best_pp = pp
            best_model = model

    print(f"Best Kneser-Ney n={n}: D={best_model.discount}, dev perplexity = {best_pp:.3f}")
    print()
    return best_model


In [ ]:
# -----------------------------
# Laplace models
# -----------------------------
laplace_bigram_model = train_laplace(n=2, train_sentences=train_sentences, vocab=vocab)
laplace_trigram_model = train_laplace(n=3, train_sentences=train_sentences, vocab=vocab)

# -----------------------------
# Kneser-Ney models
# -----------------------------
# Πρώτα bigram KN.
kn_bigram_model = train_best_kneser_ney(
    n=2,
    train_sentences=train_sentences,
    dev_sentences=dev_sentences,
    vocab=vocab
)

# Μετά trigram KN, με backoff στο bigram KN.
kn_trigram_model = train_best_kneser_ney(
    n=3,
    train_sentences=train_sentences,
    dev_sentences=dev_sentences,
    vocab=vocab,
    lower_model=kn_bigram_model
)


## 12. Έλεγχος ενδεικτικών πιθανοτήτων

Εμφανίζουμε μερικές ενδεικτικές πιθανότητες, ώστε να δούμε ότι τα μοντέλα επιστρέφουν αριθμητικές τιμές διάφορες του μηδενός.

Ελέγχουμε το:

$$
P(\text{the} \mid \text{*start*}) \quad \text{για bigram}
$$

και:

$$
P(\text{the} \mid \text{*start1*}, \text{*start2*}) \quad \text{για trigram}
$$


In [ ]:
# Bigram context αρχής πρότασης.
print("Laplace bigram P(the | *start*):", laplace_bigram_model.prob("the", (START,)))
print("Laplace bigram log P(the | *start*):", laplace_bigram_model.log_prob("the", (START,)))
print()
print("Kneser-Ney bigram P(the | *start*):", kn_bigram_model.prob("the", (START,)))
print("Kneser-Ney bigram log P(the | *start*):", kn_bigram_model.log_prob("the", (START,)))

In [ ]:
# Trigram context αρχής πρότασης.
print("Laplace trigram P(the | *start1*, *start2*):",
      laplace_trigram_model.prob("the", (START1, START2)))
print("Laplace trigram log P(the | *start1*, *start2*):",
      laplace_trigram_model.log_prob("the", (START1, START2)))
print()
print("Kneser-Ney trigram P(the | *start1*, *start2*):",
      kn_trigram_model.prob("the", (START1, START2)))
print("Kneser-Ney trigram log P(the | *start1*, *start2*):",
      kn_trigram_model.log_prob("the", (START1, START2)))


## 13. Αποτελέσματα cross-entropy και perplexity στο test set

Τώρα αξιολογούμε όλα τα μοντέλα στο test set.

Περιμένουμε συνήθως το Kneser–Ney να έχει χαμηλότερο perplexity από το Laplace, επειδή χειρίζεται καλύτερα τα sparse n-grams. Επίσης το trigram μπορεί να επηρεαστεί από sparsity, ειδικά αν το smoothing δεν είναι αρκετά καλό.


In [ ]:
# Αξιολόγηση όλων των μοντέλων στο test set.
results = []
models_to_evaluate = [
    ("Laplace", "Bigram", laplace_bigram_model),
    ("Laplace", "Trigram", laplace_trigram_model),
    ("Kneser-Ney", "Bigram", kn_bigram_model),
    ("Kneser-Ney", "Trigram", kn_trigram_model),
]

for smoothing, model_name, model in models_to_evaluate:
    H, PP = evaluate_ngram(model, test_sentences)
    results.append((smoothing, model_name, H, PP))

print(f"{'Smoothing':<12} {'Model':<8} {'Cross-entropy':>15} {'Perplexity':>15}")
print("-" * 55)
for smoothing, model_name, H, PP in results:
    print(f"{smoothing:<12} {model_name:<8} {H:>15.4f} {PP:>15.4f}")

## 14. Sentence completion / Autocomplete

Στόχος του autocomplete είναι, δοθέντος ενός prompt, να παραγάγουμε λογική συνέχεια χρησιμοποιώντας ένα από τα n-gram μοντέλα.

Σε αυτό το notebook κρατάμε **μόνο deterministic decoding methods**:

1. **Greedy decoding** — σε κάθε βήμα επιλέγουμε την πιο πιθανή επόμενη λέξη:

   $$
   \hat{w} = \arg\max_{w \in V} P(w \mid context)
   $$

2. **Beam search** — δεν κρατάμε μόνο μία υποψήφια συνέχεια, αλλά τις καλύτερες $B$ μερικές προτάσεις, ταξινομημένες με βάση:

   $$
   \text{score}(s) = \sum_{i} \log_2 P(w_i \mid context_i)
   $$

   Επειδή το beam search προτιμά μικρές προτάσεις, χρησιμοποιούμε **length normalization**:

   $$
   \text{score}_{norm}(s) = \frac{\sum_{i} \log_2 P(w_i \mid context_i)}{L^{0.7}}
   $$

   όπου $L$ είναι το πλήθος των παραγόμενων λέξεων (όχι το prompt).

Σε όλες τις μεθόδους **δεν επιτρέπουμε** να παραχθούν τα tokens: `*UNK*`, `*start*`, `*start1*`, `*start2*`.

Επιπλέον χρησιμοποιούμε έναν απλό περιορισμό `no_repeat_ngram_size`, ώστε να αποφεύγονται επαναλήψεις της μορφής:

```text
of the church of the church of the church
```

Το generation σταματά όταν παραχθεί το token `*end*` ή όταν φτάσουμε στο μέγιστο πλήθος νέων λέξεων.

### Ποιοι defs χρειάζονται για το autocomplete

Όλα όσα ακολουθούν είναι μικρές βοηθητικές συναρτήσεις, αυστηρά για greedy και beam decoding.

- **`prepare_prompt_tokens(model, prompt)`**: σπάει το prompt σε λέξεις, τις κάνει lowercase και αντικαθιστά τις OOV λέξεις με `*UNK*` ώστε να ταιριάζει με το vocabulary του μοντέλου.
- **`get_generation_context(model, generated)`**: επιστρέφει το σωστό context για το επόμενο βήμα παραγωγής — για bigram μία προηγούμενη λέξη, για trigram δύο.
- **`creates_repeated_ngram(...)`**: ελέγχει αν προσθέτοντας μια συγκεκριμένη λέξη θα δημιουργηθεί n-gram που έχει ήδη εμφανιστεί στην παραγόμενη πρόταση.
- **`get_scored_candidates(...)`**: για κάθε λέξη του vocabulary υπολογίζει $\log_2 P(w \mid context)$, αγνοεί απαγορευμένα tokens και επαναλήψεις, και επιστρέφει τη λίστα ταξινομημένη.
- **`autocomplete_decode(...)`**: η κύρια συνάρτηση, με μόνο δύο modes — `"greedy"` και `"beam"`.


In [ ]:
def prepare_prompt_tokens(model, prompt):
    """
    Μετατρέπει το prompt σε tokens και αντικαθιστά OOV λέξεις με *UNK*.

    Το prompt δεν παράγεται από το μοντέλο· είναι δοσμένο από τον χρήστη.
    """
    prompt_tokens = prompt.lower().split()
    prompt_tokens = [
        token if token in model.vocab else UNK
        for token in prompt_tokens
    ]
    return prompt_tokens


def get_generation_context(model, generated):
    """
    Επιστρέφει το σωστό context για το επόμενο βήμα παραγωγής.

    Bigram:  (προηγούμενη λέξη,)
    Trigram: (δύο προηγούμενες λέξεις)
    """
    if model.n == 2:
        if len(generated) == 0:
            return (START,)
        return (generated[-1],)

    if model.n == 3:
        if len(generated) == 0:
            return (START1, START2)
        if len(generated) == 1:
            return (START2, generated[-1])
        return (generated[-2], generated[-1])

    raise ValueError("Only bigram and trigram models are supported.")


def creates_repeated_ngram(generated, next_word, no_repeat_ngram_size=3):
    """
    Ελέγχει αν η προσθήκη της next_word δημιουργεί n-gram
    που έχει ήδη εμφανιστεί στην παραγόμενη πρόταση.
    """
    if no_repeat_ngram_size is None:
        return False
    if len(generated) + 1 < no_repeat_ngram_size:
        return False

    new_tokens = generated + [next_word]
    new_ngram = tuple(new_tokens[-no_repeat_ngram_size:])
    previous_ngrams = set(
        tuple(new_tokens[i:i + no_repeat_ngram_size])
        for i in range(len(new_tokens) - no_repeat_ngram_size)
    )
    return new_ngram in previous_ngrams


def get_scored_candidates(model, generated, no_repeat_ngram_size=3):
    """
    Για κάθε υποψήφια λέξη υπολογίζει log probability.

    Επιστρέφει λίστα από ζεύγη (log_score, word), ταξινομημένη
    από το καλύτερο score στο χειρότερο.
    """
    context = get_generation_context(model, generated)
    scored_candidates = []

    for word in sorted(model.vocab):
        # Δεν επιτρέπεται να παραχθούν UNK ή start pseudo-tokens.
        if word in {UNK, START, START1, START2}:
            continue
        # Αποφυγή επαναλαμβανόμενων n-grams.
        if creates_repeated_ngram(generated, word, no_repeat_ngram_size):
            continue
        score = model.log_prob(word, context)
        scored_candidates.append((score, word))

    scored_candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return scored_candidates


def autocomplete_decode(
    model,
    prompt,
    max_words=20,
    method="greedy",
    beam_width=5,
    no_repeat_ngram_size=3,
    return_tokens=False
):
    """
    Γενική συνάρτηση autocomplete για n-gram language models.

    method μπορεί να είναι:
      - 'greedy' : argmax σε κάθε βήμα,
      - 'beam'   : beam search με length-normalized score.
    """
    prompt_tokens = prepare_prompt_tokens(model, prompt)

    # --------------------------------------------------
    # Beam search
    # --------------------------------------------------
    if method == "beam":
        # Κάθε beam είναι: (generated_tokens, sum_log_score, finished).
        beams = [(prompt_tokens[:], 0.0, False)]

        for _ in range(max_words):
            new_beams = []
            for generated, score, finished in beams:
                if finished:
                    new_beams.append((generated, score, finished))
                    continue
                candidates = get_scored_candidates(
                    model, generated, no_repeat_ngram_size=no_repeat_ngram_size
                )
                if not candidates:
                    new_beams.append((generated, score, True))
                    continue
                # Επεκτείνουμε με κάποιους περισσότερους από beam_width υποψήφιους.
                for next_log_score, next_word in candidates[:beam_width * 5]:
                    new_generated = generated + [next_word]
                    new_score = score + next_log_score
                    new_finished = (next_word == END)
                    new_beams.append((new_generated, new_score, new_finished))

            # Length normalization: αλλιώς το beam search προτιμά πολύ μικρές προτάσεις.
            def normalized_score(beam):
                generated, score, finished = beam
                generated_len = max(1, len(generated) - len(prompt_tokens))
                return score / (generated_len ** 0.7)

            beams = sorted(new_beams, key=normalized_score, reverse=True)[:beam_width]
            if all(finished for _, _, finished in beams):
                break

        best_generated, _, _ = max(beams, key=normalized_score)
        if return_tokens:
            return best_generated
        return " ".join(best_generated)

    # --------------------------------------------------
    # Greedy decoding
    # --------------------------------------------------
    if method == "greedy":
        generated = prompt_tokens[:]
        for _ in range(max_words):
            scored_candidates = get_scored_candidates(
                model, generated, no_repeat_ngram_size=no_repeat_ngram_size
            )
            if not scored_candidates:
                break
            next_word = scored_candidates[0][1]
            generated.append(next_word)
            if next_word == END:
                break

        if return_tokens:
            return generated
        return " ".join(generated)

    raise ValueError("method must be one of: greedy, beam")


## 15. Πώς συγκρίνουμε ποιο autocomplete είναι καλύτερο;

Δεν υπάρχει ένα μόνο τέλειο αυτόματο metric για το sentence completion, γιατί δεν έχουμε μία μοναδική σωστή απάντηση. Συνδυάζουμε λοιπόν τα εξής metrics:

| Metric | Τι δείχνει | Καλύτερη τιμή |
|---|---|---:|
| `avg_log2_prob` | μέσο log probability των παραγόμενων λέξεων | μεγαλύτερο |
| `generated_ppl` | perplexity μόνο της generated συνέχειας | μικρότερο |
| `ended` | αν το μοντέλο παρήγαγε `*end*` | `True` |
| `new_tokens` | πόσες λέξεις παρήγαγε | ούτε υπερβολικά λίγες ούτε πολλές |
| `repeated_3grams` | πόσα 3-grams επαναλήφθηκαν | μικρότερο |
| `distinct_2` | ποικιλία σε bigrams | μεγαλύτερο |

### Defs αυτής της ενότητας

- **`count_repeated_ngrams(tokens, n=3)`**: μετρά επαναλήψεις. Όσο μεγαλύτερη η τιμή, τόσο πιο μονότονο το output.
- **`distinct_n(tokens, n=2)`**: λόγος μοναδικών n-grams προς συνολικά n-grams — όσο πιο κοντά στο 1, τόσο πιο ποικίλο.

  $$
  \text{distinct-}n = \frac{|\text{unique n-grams}|}{|\text{total n-grams}|}
  $$

- **`generated_continuation_score(...)`**: υπολογίζει μέσο $\log_2 P$ και perplexity **μόνο για τη συνέχεια** που παρήγαγε το μοντέλο, αγνοώντας τις λέξεις του prompt.
- **`evaluate_completion(...)`**: συγκεντρώνει όλα τα παραπάνω σε ένα dict ανά completion.
- **`compare_decoding_methods_for_prompt(...)`**: τρέχει greedy και beam για το ίδιο prompt και επιστρέφει συγκριτικό πίνακα.
- **`print_completion_comparison(rows)`**: τυπώνει αυτό τον πίνακα όμορφα.


In [ ]:
def count_repeated_ngrams(tokens, n=3):
    """Μετρά πόσα n-grams εμφανίζονται πάνω από μία φορά."""
    clean_tokens = [token for token in tokens if token != END]
    if len(clean_tokens) < n:
        return 0
    grams = [tuple(clean_tokens[i:i + n]) for i in range(len(clean_tokens) - n + 1)]
    counts = Counter(grams)
    return sum(count - 1 for count in counts.values() if count > 1)


def distinct_n(tokens, n=2):
    """Distinct-n = unique n-grams / total n-grams."""
    clean_tokens = [token for token in tokens if token != END]
    if len(clean_tokens) < n:
        return 0.0
    grams = [tuple(clean_tokens[i:i + n]) for i in range(len(clean_tokens) - n + 1)]
    return len(set(grams)) / len(grams)


def generated_continuation_score(model, prompt, generated_tokens):
    """
    Υπολογίζει μέσο log2 prob και perplexity μόνο για τη συνέχεια
    (όχι για τις λέξεις του prompt).
    """
    prompt_tokens = prepare_prompt_tokens(model, prompt)
    prompt_len = len(prompt_tokens)

    if model.n == 2:
        full_tokens = [START] + generated_tokens
        first_predicted_index = 1 + prompt_len
    else:
        full_tokens = [START1, START2] + generated_tokens
        first_predicted_index = 2 + prompt_len

    total_log_prob = 0.0
    total_tokens = 0

    for i in range(first_predicted_index, len(full_tokens)):
        context = tuple(full_tokens[i - model.n + 1:i])
        word = full_tokens[i]
        if word in {START, START1, START2}:
            continue
        total_log_prob += model.log_prob(word, context)
        total_tokens += 1

    if total_tokens == 0:
        return 0.0, float("inf")
    avg_log2_prob = total_log_prob / total_tokens
    generated_ppl = 2 ** (-avg_log2_prob)
    return avg_log2_prob, generated_ppl


def evaluate_completion(model, prompt, generated_tokens):
    """Επιστρέφει αυτόματα metrics για ένα generated completion."""
    prompt_tokens = prepare_prompt_tokens(model, prompt)
    continuation = generated_tokens[len(prompt_tokens):]
    avg_log2_prob, generated_ppl = generated_continuation_score(model, prompt, generated_tokens)
    return {
        "avg_log2_prob": avg_log2_prob,
        "generated_ppl": generated_ppl,
        "ended": len(generated_tokens) > 0 and generated_tokens[-1] == END,
        "new_tokens": len(continuation),
        "repeated_3grams": count_repeated_ngrams(generated_tokens, n=3),
        "distinct_2": distinct_n(generated_tokens, n=2),
    }


def compare_decoding_methods_for_prompt(model, prompt):
    """
    Τρέχει greedy και beam για ένα prompt και επιστρέφει rows αποτελεσμάτων.
    """
    method_configs = [
        ("greedy", {"method": "greedy"}),
        ("beam",   {"method": "beam", "beam_width": 5}),
    ]
    rows = []
    for method_name, config in method_configs:
        generated_tokens = autocomplete_decode(
            model,
            prompt,
            max_words=20,
            no_repeat_ngram_size=3,
            return_tokens=True,
            **config
        )
        metrics = evaluate_completion(model, prompt, generated_tokens)
        rows.append({
            "method": method_name,
            "text": " ".join(generated_tokens),
            **metrics
        })
    return rows


def print_completion_comparison(rows):
    """Εκτυπώνει συγκριτικό πίνακα autocomplete."""
    print(
        f"{'Method':<10} {'Ended':<6} {'Len':>4} {'AvgLog2':>10} "
        f"{'GenPPL':>10} {'Rep3':>5} {'Dist2':>7}  Text"
    )
    print("-" * 120)
    for row in rows:
        print(
            f"{row['method']:<10} {str(row['ended']):<6} {row['new_tokens']:>4} "
            f"{row['avg_log2_prob']:>10.3f} {row['generated_ppl']:>10.2f} "
            f"{row['repeated_3grams']:>5} {row['distinct_2']:>7.3f}  {row['text']}"
        )


## 16. Grid search για το beam width

Τα language models αξιολογούνται με **cross-entropy** και **perplexity** στο test set. Το **autocomplete** όμως είναι διαφορετικό πρόβλημα: δεν υπάρχει μία σωστή συνέχεια για ένα prompt. Για αυτό κάνουμε **grid search στο development set** και επιλέγουμε το καλύτερο deterministic setting.

Επειδή κρατάμε μόνο greedy και beam:

- το **greedy** δεν έχει υπερπαραμέτρους,
- το **beam search** έχει μόνο μία υπερπαράμετρο: το `beam_width` ∈ {3, 5, 8}.

Για τη σύγκριση χρησιμοποιούμε heuristic score:

$$
\text{score} = \overline{\log_2 P} + 1.5 \cdot \text{ended} + 0.8 \cdot \text{distinct\_2} - 0.5 \cdot \text{repeated\_3grams} - 0.03 \cdot |new\_tokens - 12|
$$

όπου:

- $\overline{\log_2 P}$: μέση log πιθανότητα των generated tokens,
- `ended`: αν η συνέχεια τελείωσε με `*end*`,
- `distinct_2`: ποικιλία στα bigrams του generated text,
- `repeated_3grams`: πόσες επαναλήψεις trigram υπάρχουν,
- `new_tokens`: μήκος της συνέχειας.

Το score δεν αντικαθιστά την ανθρώπινη αξιολόγηση fluency. Το χρησιμοποιούμε ως πρακτικό κριτήριο για να διαλέξουμε καλύτερες υπερπαραμέτρους και πιο καθαρά examples για το report.

### Defs αυτής της ενότητας

- **`make_decoding_dev_prompts(...)`**: εξάγει prompts από το dev set (όχι test, ώστε να αποφεύγεται data leakage).
- **`completion_quality_score(metrics)`**: συνδυάζει τα metrics στο παραπάνω heuristic score.
- **`decoding_grid_configs()`**: επιστρέφει τη λίστα settings που θα δοκιμαστούν — μόνο greedy και beam με $B \in \{3, 5, 8\}$.
- **`evaluate_decoding_config(...)`**: τρέχει ένα setting σε όλα τα dev prompts και συλλέγει metrics.
- **`average_metric(rows, key)`**: μέσος όρος ενός metric πάνω σε όλα τα rows.
- **`tune_decoding_for_model(...)`**: τρέχει όλα τα settings για ένα μοντέλο και κρατά το καλύτερο.


In [ ]:
def make_decoding_dev_prompts(sentences, max_prompts=10, prefix_len=3):
    """Φτιάχνει prompts από το dev set (όχι test)."""
    prompts = []
    for sent in sentences:
        if len(sent) >= prefix_len + 2 and UNK not in sent[:prefix_len]:
            prompts.append(" ".join(sent[:prefix_len]))
        if len(prompts) >= max_prompts:
            break
    return prompts


def completion_quality_score(metrics, target_length=12):
    """Heuristic score για completion quality."""
    return (
        metrics["avg_log2_prob"]
        + 1.5 * int(metrics["ended"])
        + 0.8 * metrics["distinct_2"]
        - 0.5 * metrics["repeated_3grams"]
        - 0.03 * abs(metrics["new_tokens"] - target_length)
    )


def decoding_grid_configs():
    """Όλα τα deterministic decoding settings που θα δοκιμαστούν."""
    configs = []
    configs.append(("greedy", {"method": "greedy"}))
    for beam_width in [3, 5, 8]:
        configs.append((
            f"beam_w{beam_width}",
            {"method": "beam", "beam_width": beam_width}
        ))
    return configs


def evaluate_decoding_config(model, prompts, config, max_words=20):
    """Αξιολογεί ένα decoding setting σε όλα τα dev prompts."""
    rows = []
    for prompt in prompts:
        generated_tokens = autocomplete_decode(
            model,
            prompt,
            max_words=max_words,
            no_repeat_ngram_size=3,
            return_tokens=True,
            **config
        )
        metrics = evaluate_completion(model, prompt, generated_tokens)
        metrics["quality_score"] = completion_quality_score(metrics)
        metrics["prompt"] = prompt
        metrics["text"] = " ".join(generated_tokens)
        rows.append(metrics)
    return rows


def average_metric(rows, key):
    """Μέσος όρος ενός metric σε λίστα από rows."""
    return sum(row[key] for row in rows) / len(rows)


def tune_decoding_for_model(model_name, model, prompts):
    """Grid search για ένα μοντέλο. Επιστρέφει (όλα τα summaries, καλύτερο setting)."""
    summaries = []
    for config_name, config in decoding_grid_configs():
        rows = evaluate_decoding_config(model, prompts, config)
        summary = {
            "model": model_name,
            "config_name": config_name,
            "config": config,
            "avg_quality_score": average_metric(rows, "quality_score"),
            "avg_log2_prob": average_metric(rows, "avg_log2_prob"),
            "avg_generated_ppl": average_metric(rows, "generated_ppl"),
            "end_rate": average_metric(rows, "ended"),
            "avg_new_tokens": average_metric(rows, "new_tokens"),
            "avg_repeated_3grams": average_metric(rows, "repeated_3grams"),
            "avg_distinct_2": average_metric(rows, "distinct_2"),
        }
        summaries.append(summary)
    summaries.sort(key=lambda row: row["avg_quality_score"], reverse=True)
    best = summaries[0]
    return summaries, best


# Μικρό dev set από prompts για γρήγορο tuning.
decoding_dev_prompts = make_decoding_dev_prompts(dev_sentences, max_prompts=10, prefix_len=3)

print("Dev prompts που χρησιμοποιούνται για decoding tuning:")
for prompt in decoding_dev_prompts:
    print("-", prompt)

# Κάνουμε tuning για όλα τα μοντέλα.
decoding_models = [
    ("Laplace Bigram",  laplace_bigram_model),
    ("Laplace Trigram", laplace_trigram_model),
    ("KN Bigram",       kn_bigram_model),
    ("KN Trigram",      kn_trigram_model),
]

all_decoding_summaries = []
best_decoding_settings = {}

for model_name, model in decoding_models:
    summaries, best = tune_decoding_for_model(model_name, model, decoding_dev_prompts)
    all_decoding_summaries.extend(summaries)
    best_decoding_settings[model_name] = best

print("\nΚαλύτερο decoding setting ανά μοντέλο:")
print(
    f"{'Model':<18} {'Best config':<12} {'Score':>10} {'GenPPL':>10} "
    f"{'EndRate':>8} {'Rep3':>8} {'Dist2':>8}"
)
print("-" * 85)
for model_name, best in best_decoding_settings.items():
    print(
        f"{model_name:<18} "
        f"{best['config_name']:<12} "
        f"{best['avg_quality_score']:>10.3f} "
        f"{best['avg_generated_ppl']:>10.2f} "
        f"{best['end_rate']:>8.2f} "
        f"{best['avg_repeated_3grams']:>8.2f} "
        f"{best['avg_distinct_2']:>8.3f}"
    )


## 17. Τελικά autocomplete examples με τα καλύτερα tuned settings

Στο προηγούμενο cell έγινε grid search στο development set. Τώρα χρησιμοποιούμε **αυτόματα το καλύτερο decoding setting ανά μοντέλο** (greedy ή beam με το βέλτιστο `beam_width`) για να παραγάγουμε τα τελικά examples που μπαίνουν στο report.

### Defs αυτής της ενότητας

- **`generate_with_best_decoding(model_name, model, prompt)`**: φορτώνει το αποθηκευμένο best setting του μοντέλου και τρέχει `autocomplete_decode`. Επιστρέφει τα generated tokens μαζί με τα metrics τους.


In [ ]:
def generate_with_best_decoding(model_name, model, prompt):
    """
    Παράγει completion χρησιμοποιώντας το καλύτερο decoding setting
    που βρέθηκε από το grid search για το συγκεκριμένο μοντέλο.
    """
    best = best_decoding_settings[model_name]
    config = dict(best["config"])

    generated_tokens = autocomplete_decode(
        model,
        prompt,
        max_words=20,
        no_repeat_ngram_size=3,
        return_tokens=True,
        **config
    )
    metrics = evaluate_completion(model, prompt, generated_tokens)
    metrics["quality_score"] = completion_quality_score(metrics)
    return generated_tokens, metrics, best


final_prompts = [
    "the man",
    "in the",
    "he was",
    "she had",
    "the old",
    "i would like to commend the",
]

final_example_rows = []

for prompt in final_prompts:
    print("=" * 120)
    print("Prompt:", prompt)
    print("=" * 120)
    for model_name, model in decoding_models:
        generated_tokens, metrics, best = generate_with_best_decoding(model_name, model, prompt)
        text = " ".join(generated_tokens)
        final_example_rows.append({
            "prompt": prompt,
            "model": model_name,
            "best_config": best["config_name"],
            "text": text,
            **metrics,
        })
        print(f"{model_name:<18} [{best['config_name']}]: {text}")
        print(
            f"  score={metrics['quality_score']:.3f}, "
            f"ppl={metrics['generated_ppl']:.2f}, "
            f"ended={metrics['ended']}, "
            f"rep3={metrics['repeated_3grams']}, "
            f"dist2={metrics['distinct_2']:.3f}"
        )
    print()


### Αυτόματη επιλογή καλών και κακών examples για το report

Η εκφώνηση ζητά να δείξουμε ενδιαφέρουσες περιπτώσεις — δηλαδή καλά και κακά completions. Ταξινομούμε τα τελικά examples με βάση το heuristic score και κρατάμε:

- τα καλύτερα examples,
- τα χειρότερα examples,
- ένα ενδεικτικό καλό trigram example (που τελειώνει με `*end*` και δεν έχει επαναλήψεις),
- ένα ενδεικτικό κακό bigram ή greedy-like example.

Στο report χρησιμοποιούμε αυτά τα outputs ως συμπεράσματα, μαζί με σύντομο σχολιασμό.


In [ ]:
# Ταξινόμηση final examples με βάση το quality score.
examples_sorted = sorted(
    final_example_rows,
    key=lambda row: row["quality_score"],
    reverse=True
)

best_examples_for_report = examples_sorted[:5]
worst_examples_for_report = examples_sorted[-5:]

print("Καλύτερα examples για το report:")
for row in best_examples_for_report:
    print(
        f"[{row['model']} | {row['best_config']} | score={row['quality_score']:.3f}] "
        f"Prompt: {row['prompt']} -> {row['text']}"
    )

print("\nΧειρότερα / πιο προβληματικά examples για το report:")
for row in worst_examples_for_report:
    print(
        f"[{row['model']} | {row['best_config']} | score={row['quality_score']:.3f}] "
        f"Prompt: {row['prompt']} -> {row['text']}"
    )

# Ένα καλό trigram example και ένα προβληματικό example.
good_trigram_examples = [
    row for row in examples_sorted
    if "Trigram" in row["model"] and row["ended"] and row["repeated_3grams"] == 0
]
bad_examples = [
    row for row in reversed(examples_sorted)
    if "Bigram" in row["model"] or row["repeated_3grams"] > 0
]

print("\nΠροτεινόμενο καλό trigram example:")
if good_trigram_examples:
    row = good_trigram_examples[0]
    print(f"Prompt: {row['prompt']}")
    print(f"Model: {row['model']} / {row['best_config']}")
    print(row["text"])
else:
    print("Δεν βρέθηκε trigram example που να τελειώνει με *end* και να μην έχει repeated trigrams.")

print("\nΠροτεινόμενο προβληματικό example:")
if bad_examples:
    row = bad_examples[0]
    print(f"Prompt: {row['prompt']}")
    print(f"Model: {row['model']} / {row['best_config']}")
    print(row["text"])
else:
    print("Δεν βρέθηκε εμφανώς προβληματικό example στα τελικά rows.")

## 18. Παραδείγματα autocomplete: greedy vs beam

Παρακάτω δείχνουμε outputs και από τα δύο decoding methods (greedy και beam) για κάθε μοντέλο.

Για καθαρή σύγκριση κρατάμε τα ίδια prompts και κοιτάμε κυρίως:

- αν το **trigram** είναι πιο φυσικό από το **bigram**,
- αν το **Kneser–Ney** αποφεύγει καλύτερα τις πολύ κακές sparse εκτιμήσεις,
- αν το **beam search** βρίσκει υψηλότερου probability ακολουθίες σε σχέση με greedy.


In [ ]:
# Παραδείγματα prompts για autocomplete.
prompts = [
    "the man",
    "in the",
    "he was",
    "she had",
    "the old"
]

generation_models = [
    ("Laplace Bigram",  laplace_bigram_model),
    ("Laplace Trigram", laplace_trigram_model),
    ("KN Bigram",       kn_bigram_model),
    ("KN Trigram",      kn_trigram_model),
]

for prompt in prompts:
    print("=" * 120)
    print("Prompt:", prompt)
    print("=" * 120)
    for model_name, model in generation_models:
        print(f"\nModel: {model_name}")
        rows = compare_decoding_methods_for_prompt(model, prompt)
        print_completion_comparison(rows)

## 19. Πώς επιλέγουμε το «καλύτερο» completion method;

Για την εργασία δεν χρειάζεται να αποδείξουμε ότι ένα decoding method είναι πάντα καλύτερο. Χρειάζεται να δείξουμε παραδείγματα και να εξηγήσουμε τι παρατηρούμε.

Ένας πρακτικός τρόπος επιλογής είναι:

1. **Απορρίπτουμε outputs με πολλές επαναλήψεις** — αν το `repeated_3grams` είναι μεγάλο, το completion είναι μάλλον κακό.
2. **Προτιμάμε outputs που τελειώνουν με `*end*`** — αν `ended=True`, το μοντέλο κατάφερε να ολοκληρώσει την πρόταση.
3. **Κοιτάμε το generated perplexity** — μικρότερο `generated_ppl` σημαίνει ότι η συνέχεια είναι πιο πιθανή σύμφωνα με το ίδιο το μοντέλο.
4. **Κοιτάμε diversity** — μεγαλύτερο `distinct_2` σημαίνει λιγότερη μονοτονία.
5. **Κάνουμε τελική ανθρώπινη κρίση** — το σημαντικότερο για το autocomplete είναι αν το output διαβάζεται φυσικά.

Στα deterministic μοντέλα που κρατάμε:

- το **greedy** είναι απλό και αναπαραγώγιμο, αλλά συχνά κολλάει σε επαναλήψεις,
- το **beam search** βρίσκει συχνά πιο πιθανές ακολουθίες, ειδικά με length normalization, αλλά μπορεί να παράγει επίσης μονότονα completions όταν το beam συγκλίνει σε όμοια paths.


In [ ]:
# Συνοπτική σύγκριση μόνο για KN Trigram, επειδή συνήθως είναι το πιο ενδιαφέρον μοντέλο.
summary_rows = []
for prompt in prompts:
    rows = compare_decoding_methods_for_prompt(kn_trigram_model, prompt)
    for row in rows:
        summary_rows.append({"prompt": prompt, **row})

print(
    f"{'Prompt':<10} {'Method':<10} {'Ended':<6} {'Len':>4} "
    f"{'GenPPL':>10} {'Rep3':>5} {'Dist2':>7}  Text"
)
print("-" * 130)
for row in summary_rows:
    print(
        f"{row['prompt']:<10} {row['method']:<10} {str(row['ended']):<6} "
        f"{row['new_tokens']:>4} {row['generated_ppl']:>10.2f} "
        f"{row['repeated_3grams']:>5} {row['distinct_2']:>7.3f}  {row['text']}"
    )

## 20. Σχόλια για τα αποτελέσματα

Μετά την εκτέλεση του notebook, ο πρώτος συγκριτικός πίνακας δείχνει το **cross-entropy** και το **perplexity** για κάθε συνδυασμό μοντέλου/smoothing. Αυτά είναι τα βασικά αποτελέσματα για την αξιολόγηση των language models.

Για το autocomplete έγινε ξεχωριστό **grid search στο development set** για τη μοναδική υπερπαράμετρο που έχουμε — το `beam_width`. Τα καλύτερα settings επιλέγονται με heuristic score που συνδυάζει μέση log πιθανότητα, σωστό τέλος με `*end*`, λιγότερες επαναλήψεις και μεγαλύτερη ποικιλία. Έτσι μπορούμε να πούμε ξεκάθαρα ότι οι υπερπαράμετροι του autocomplete δεν επιλέχθηκαν τυχαία.

Τα βασικά συμπεράσματα είναι:

1. Το **bigram** συχνά κολλάει σε τοπικά μοτίβα, επειδή βλέπει μόνο μία προηγούμενη λέξη.
2. Το **trigram** συνήθως παράγει πιο φυσικές φράσεις, επειδή χρησιμοποιεί context δύο λέξεων.
3. Το **greedy decoding** είναι απλό, αλλά μπορεί να οδηγήσει σε επαναλήψεις.
4. Το **beam search** με length-normalized score συνήθως βρίσκει υψηλότερου probability ακολουθίες και αξιοποιεί καλύτερα το context.

Για το report κρατάμε τα examples από την ενότητα «Τελικά autocomplete examples με τα καλύτερα tuned settings» και ειδικά τα rows που εμφανίζονται στην «Αυτόματη επιλογή καλών και κακών examples». Έτσι καλύπτεται το ζητούμενο της εκφώνησης για **good and bad sentence completions**.


## 21. Τελικό συμπέρασμα

Σε αυτό το notebook υλοποιήθηκαν bigram και trigram language models με δύο διαφορετικές μεθόδους smoothing:

- **Laplace smoothing**,
- **Kneser–Ney smoothing**.

Η χρήση της βασικής κλάσης `NGramLM` κρατάει κοινό τον n-gram μηχανισμό, ενώ οι υποκλάσεις `LaplaceLM` και `KneserNeyLM` αλλάζουν μόνο τον τρόπο υπολογισμού της πιθανότητας:

$$
P(w \mid context)
$$

Έτσι η σύγκριση Laplace και Kneser–Ney είναι καθαρή και συμβατή με την εκφώνηση της εργασίας.

Για την αξιολόγηση των μοντέλων χρησιμοποιούνται:

$$
H = -\frac{1}{N}\sum_{i=1}^{N}\log_2 P(w_i \mid context_i)
$$

και:

$$
PP = 2^{H}
$$

Για το autocomplete χρησιμοποιήθηκαν δύο deterministic decoding methods:

- **greedy decoding**, και
- **beam search** με length normalization.

Με αυτή την επιλογή τα completions είναι **πλήρως αναπαραγώγιμα** και η σύγκριση των μοντέλων δεν εξαρτάται από τυχαιότητα στη δειγματοληψία.

Επιπλέον, έγινε **grid search** πάνω στο development set για το `beam_width` ∈ {3, 5, 8}, ώστε να επιλεγούν τα καλύτερα settings ανά μοντέλο.

Η τελική ερμηνεία είναι ότι το **perplexity** συγκρίνει τα language models, ενώ τα autocomplete examples συγκρίνουν διαφορετικές στρατηγικές decoding. Τα trigram μοντέλα, ειδικά με Kneser–Ney smoothing και tuned beam search, αναμένεται να δίνουν πιο φυσικές συνέχειες από τα bigram μοντέλα, επειδή αξιοποιούν μεγαλύτερο context και καλύτερο smoothing για sparse n-grams.
